In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT = r"H:\Compatifi Model\V5A_Final_Merged_Model"

# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5A MODEL")
print("=" * 100)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

print("Loading merged model...")
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print("✅ V5A Model Loaded Successfully")


# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V5A.

Your task is to determine what the assistant should try to accomplish
in the current conversation.

V5A predicts the assistant's immediate objective.

It does NOT predict:
- the user's long-term goals
- memories
- personality
- emotions as memories
- a response
- future events

Use ONLY:
- Domain
- Relationship
- Conversation

Always predict ONE primary objective.

The secondary objective may be "None" if unnecessary.

Valid primary objectives include:
- Emotional Support
- Reduce Anxiety
- Solve Problem
- Decision Support
- Planning Assistance
- Motivation
- Information Sharing
- Maintain Rapport

Valid secondary objectives include:
- None
- Build Confidence
- Clarify Situation
- Encourage Reflection
- Suggest Next Steps
- Maintain Rapport

Priority must be:
- High
- Medium
- Low

Return ONLY valid JSON.

Required format:

{
  "primary_objective": "...",
  "secondary_objective": "...",
  "priority": "...",
  "reason": "..."
}

The reason must be based ONLY on information explicitly present
in the conversation.

Do not generate a reply to the user.
"""



g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


LOADING V5A MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 4/4 [01:21<00:00, 20.29s/it]


✅ V5A Model Loaded Successfully


In [14]:

# ==========================================================
# TEST CONVERSATION
# ==========================================================



test_conversations = [

    {
        "name": "SECONDARY OBJECTIVE TEST - Interview Anxiety",
        "domain": "career",
        "relationship": "Mentor",

        "conversation": """
Mentor: How are you feeling about your interview tomorrow?

User: Honestly, I'm really nervous. I know I've prepared well,
but I keep thinking I'm going to fail.

Mentor: What makes you feel that way?

User: I keep doubting whether I'm good enough for the position.

Mentor: Your preparation has been strong, and you've already
practiced the difficult questions several times.

User: I know, but I still don't feel confident.

Mentor: Then let's focus on reminding you of what you've already
accomplished and preparing you to approach the interview calmly.

User: Yes, I think I need that.
"""
    }

]



In [15]:

# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    user_prompt = f"""
Domain: {test['domain']}

Relationship: {test['relationship']}

Conversation:

{test['conversation']}

Instruction:
Predict the assistant objective.
"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    # ------------------------------------------------------
    # DECODE ONLY NEW TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    # ------------------------------------------------------
    # PRINT RESULT
    # ------------------------------------------------------

    print("\nMODEL OUTPUT")
    print("-" * 100)
    print(response)

    print("\n")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




SECONDARY OBJECTIVE TEST - Interview Anxiety

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"primary_objective":"Reduce Anxiety","secondary_objective":"None","priority":"High","reason":"The user is feeling very nervous and doubting their readiness for an upcoming interview.","memories":null}


